# ETF Document Loader

Load ETF/mutual fund metadata into ChromaDB using the parser factory.
Each fund family has a dedicated parser; sources are activated via `FUND_SOURCES` below.

In [1]:
%reload_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.etf.etf_document_loader import ETFDocumentLoader
from apps.agentic.core.document_loaders.etf.etf_parser_factory import ETFParserFactory
from apps.agentic.core.constants import ETF_DB_NAME, ETF_COLLECTION_NAME
from apps.agentic.core.utils import set_chatgpt_env, set_tavily_env

DB_PATH = Path("../../.db").resolve()
DB_PATH.mkdir(parents=True, exist_ok=True)

set_chatgpt_env(filedir="../../.keys")
set_tavily_env(filedir="../../.keys")

## Configuration

In [2]:
ETF_DB_NAME, ETF_COLLECTION_NAME, DB_PATH

('etf', 'etf-info', PosixPath('/Users/troy/Develop/gly.fish/yada/.db'))

## Load funds via factory

In [3]:
# Map each source key to its file path.
# Comment/uncomment sources to control which funds are loaded.
FUND_SOURCES = {
    "vaneck":      "../../market_data/ETFs/VanEck-ETFs.xls",
    # "wisdomtree":  "../../market_data/ETFs/WisdomTree-ETF.xlsx",
    # "statestreet": "../../market_data/ETFs/StateStreet-ETFs.xlsx",
    # "vanguard":    "../../market_data/ETFs/Vanguard-ETFs.csv",
    # "proshares":   "../../market_data/ETFs/Proshares-ETFs-performance.csv",
    # "ishares":     "../../market_data/ETFs/Blackrock-etfs.md",
}

LOAD_LIMIT = None  # set to an int (e.g. 10) to cap records per source during testing

FUND_SOURCES

{'vaneck': '../../market_data/ETFs/VanEck-ETFs.xls'}

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_results=3, include_answer=True)

for source, path in FUND_SOURCES.items():
    print(f"Loading {source} from {path} ...")
    parser = ETFParserFactory.get(source)
    loader = ETFDocumentLoader(parser=parser, tavily_client=tavily, db_path=str(DB_PATH))
    await loader.reload_document(path, fund_family=parser.FUND_FAMILY, limit=LOAD_LIMIT)
    count = loader.vectorstore._collection.count()
    print(f"  → collection now has {count} docs")

In [5]:
# Inspect what's in the collection — reuse the loader's already-open vectorstore
coll = loader.vectorstore._collection
print("Total docs:", coll.count())

probe = coll.get(limit=5)
docs: list = probe.get("documents") or []
metas: list = probe.get("metadatas") or []
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"\n--- [{i}] {meta.get('ticker')} | {meta.get('fund_family')} ---")
    print(doc[:500])

Total docs: 90

--- [0] AFK | VanEck ---
# VanEck Africa Index ETF

- **Ticker:** AFK
- **Fund Family:** VanEck
- **Asset Class:** Equity
- **Category:** International
- **Inception Date:** 2008-07-10
- **AUM:** $108,500,000
- **Expense Ratio:** N/A
- **Benchmark:** N/A

## Description
The VanEck Africa Index ETF (AFK) aims to mirror the MVIS GDP Africa Index, investing in companies with significant African operations. It typically allocates at least 80% of its assets to the index's securities. The fund's performance is evaluated based

--- [1] ANGL | VanEck ---
# VanEck Fallen Angel High Yield Bond ETF

- **Ticker:** ANGL
- **Fund Family:** VanEck
- **Asset Class:** Income
- **Category:** Corporate Bonds
- **Inception Date:** 2012-04-10
- **AUM:** $3,100,000,000
- **Expense Ratio:** N/A
- **Benchmark:** N/A

## Description
The VanEck Fallen Angel High Yield Bond ETF (ANGL) targets fallen angel bonds, which were once investment grade but are now below investment grade, aiming for stron

In [6]:
# Similarity search across the loaded funds
results = loader.vectorstore.similarity_search("international equity dividend growth", k=5)
for r in results:
    print(r.metadata.get("ticker"), "|", r.metadata.get("fund_name"))
    print(r.page_content[:300])
    print()

GLIN | VanEck India Growth Leaders ETF
# VanEck India Growth Leaders ETF

- **Ticker:** GLIN
- **Fund Family:** VanEck
- **Asset Class:** Equity
- **Category:** International
- **Inception Date:** 2010-08-24
- **AUM:** $95,100,000
- **Expense Ratio:** N/A
- **Benchmark:** N/A

## Description
The VanEck India Growth Leaders ETF (GLIN) aim

EMRIX | VanEck Emerging Markets Fund-I
# VanEck Emerging Markets Fund-I

- **Ticker:** EMRIX
- **Fund Family:** VanEck
- **Asset Class:** Equity
- **Category:** International
- **Inception Date:** 2007-12-31
- **AUM:** $36,700,000
- **Expense Ratio:** N/A
- **Benchmark:** N/A

## Description
The VanEck Emerging Markets Fund-I ETF primari

EMBX | VanEck Emerging Markets Bond ETF
# VanEck Emerging Markets Bond ETF

- **Ticker:** EMBX
- **Fund Family:** VanEck
- **Asset Class:** Income
- **Category:** International Bond
- **Inception Date:** 2012-07-09
- **AUM:** $239,500,000
- **Expense Ratio:** N/A
- **Benchmark:** N/A

## Description
The VanEck Emergi